In [19]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go

a, b = -1, 1
x0, sig = 0, 0.5
prc = 100

A1 = np.random.uniform(low=a, high=b, size=prc)
B1 = np.random.uniform(low= a, high= b, size=prc)
E1 = np.random.uniform(low=a, high=b, size=prc)
A2 = np.random.normal(x0, sig, prc)
B2 = np.random.normal(x0, sig, prc)
E2 = np.random.normal(x0, sig, prc)

X1 = np.column_stack((A1, B1 + 0.5, A1 * E1))
X2 = np.column_stack((A2, B2 + E2, 0.5 + A2 + E2))

X = np.concatenate([X1, X2])

fig = go.Figure()
fig.add_trace(go.Scatter3d(x=X1[:, 0], y=X1[:, 1], z=X1[:, 2], mode='markers', marker=dict(size=3), name='X1'))
fig.add_trace(go.Scatter3d(x=X2[:, 0], y=X2[:, 1], z=X2[:, 2], mode='markers', marker=dict(size=3), name='X2'))
fig.update_layout(height=700, width=900, margin=dict(l=0, r=0, t=0, b=0))
fig.show()

In [20]:
from sklearn.metrics import pairwise_distances
from sklearn.cluster import AffinityPropagation
S = -pairwise_distances(X, metric='sqeuclidean')
med = np.median(S)
print(med)
for k in [0.5, 1, 2, 5, 10, 20, 25]:
    ap = AffinityPropagation(preference=med * k, random_state=0).fit(X)
    print(k, len(ap.cluster_centers_indices_))

-1.7232365872109785
0.5 23
1 15
2 11
5 8
10 5
20 3
25 2


In [29]:
from sklearn.cluster import AffinityPropagation

ap = AffinityPropagation(damping = 0.5, random_state=0, max_iter = 200, preference = med * 3)
ap.fit(X)

,"preference preference: array-like of shape (n_samples,) or float, default=NonePreferences for each point - points with larger values ofpreferences are more likely to be chosen as exemplars. The numberof exemplars, ie of clusters, is influenced by the inputpreferences value. If the preferences are not passed as arguments,they will be set to the median of the input similarities.",np.float64(-5...7097616329355)
,"random_state random_state: int, RandomState instance or None, default=NonePseudo-random number generator to control the starting state.Use an int for reproducible results across function calls.See the :term:`Glossary <random_state>`... versionadded:: 0.23 this parameter was previously hardcoded as 0.",0
,"damping damping: float, default=0.5Damping factor in the range `[0.5, 1.0)` is the extent towhich the current value is maintained relative toincoming values (weighted 1 - damping). This in orderto avoid numerical oscillations when updating thesevalues (messages).",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations.",200
,"convergence_iter convergence_iter: int, default=15Number of iterations with no change in the numberof estimated clusters that stops the convergence.",15
,"copy copy: bool, default=TrueMake a copy of input data.",True
,"affinity affinity: {'euclidean', 'precomputed'}, default='euclidean'Which affinity to use. At the moment 'precomputed' and``euclidean`` are supported. 'euclidean' uses thenegative squared euclidean distance between points.",'euclidean'
,"verbose verbose: bool, default=FalseWhether to be verbose.",False
Name,Type,Value
"affinity_matrix_ affinity_matrix_: ndarray of shape (n_samples, n_samples)Stores the affinity matrix used in ``fit``.","ndarray[float64](200, 200)","[[-5.17,-0.96,-1.45,...,-3.83,-0.76,-0.41], [-0.96,-5.17,-3.83,...,-6.96,-2.07,-2.3 ], [-1.45,-3.83,-5.17,...,-2.89,-1.84,-1.56], ..., [-3.83,-6.96,-2.89,...,-5.17,-6.81,-4.57], [-0.76,-2.07,-1.84,...,-6.81,-5.17,-0.37], [-0.41,-2.3 ,-1.56,...,-4.57,-0.37,-5.17]]"
"cluster_centers_ cluster_centers_: ndarray of shape (n_clusters, n_features)Cluster centers (if affinity != ``precomputed``).","ndarray[float64](8, 3)","[[ 0.56, 0.34,-0.03], [ 0.74,-0.49, 0.73], [ 0.46, 1.25, 0.28], ..., [ 0.55, 0.64, 1.48], [ 0.17,-1.38,-0.14], [-0.41,-0.3 ,-0.09]]"


In [30]:
from sklearn.metrics import rand_score, adjusted_rand_score
from sklearn.metrics import silhouette_score
lab = ap.predict(X)
#Как считается Rand Index. Берутся все пары точек, и для каждой пары проверяется, согласны ли два разбиения. Пара засчитывается как совпадение в двух случаях: обе точки вместе и в y_true, и в lab, или обе точки порознь и там, и там. RI — это доля таких пар, от 0 до 1, где 1 означает полное совпадение

y_tr = np.array([0] * len(X1) + [1] * len(X2))

ri = rand_score(y_tr, lab)
ari = adjusted_rand_score(y_tr, lab)
sil = silhouette_score(X, lab) # s_i = (b_i - a_i) / max(b_i, a_i), a_i - среднее расстояние до объектово своего кластера, b_i - до чужого

C = ap.cluster_centers_             # координаты центров (красные)

# линии от точки к её центру
lx, ly, lz = [], [], []
for p, c in zip(X, C[lab]):
    lx += [p[0], c[0], None]
    ly += [p[1], c[1], None]
    lz += [p[2], c[2], None]

# наконечники стрелок у центра
d = C[lab] - X
L = np.linalg.norm(d, axis=1)
m = L > 1e-9                        # сами центры стрелку не рисуют
u = d[m] / L[m, None]
tip = X[m] + d[m] * 0.85

fig = go.Figure()
fig.add_trace(go.Scatter3d(x=lx, y=ly, z=lz, mode='lines',
                           line=dict(color='black', width=2), hoverinfo='skip'))
fig.add_trace(go.Cone(x=tip[:, 0], y=tip[:, 1], z=tip[:, 2],
                      u=u[:, 0], v=u[:, 1], w=u[:, 2],
                      sizemode='absolute', sizeref=0.06, anchor='tip',
                      colorscale=[[0, 'black'], [1, 'black']], showscale=False))
fig.add_trace(go.Scatter3d(x=X[:, 0], y=X[:, 1], z=X[:, 2], mode='markers',
                           marker=dict(size=4, color='limegreen', line=dict(color='darkgreen', width=1))))
fig.add_trace(go.Scatter3d(x=C[:, 0], y=C[:, 1], z=C[:, 2], mode='markers',
                           marker=dict(size=9, color='red', line=dict(color='darkred', width=1))))

fig.update_layout(height=700, width=900, margin=dict(l=0, r=0, t=0, b=0), showlegend=False)
fig.update_layout(title=f'RI = {ri:.3f}   ARI = {ari:.3f}   Silhouette  = {sil:.3f}', margin=dict(t=40))
fig.show()

In [12]:
def ap_history(X, damping=0.5, max_iter=200, convergence_iter=15):
    n = len(X)
    S = -((X[:, None] - X[None]) ** 2).sum(-1)       # сходства
    np.fill_diagonal(S, np.median(S))                 # preference
    R = np.zeros((n, n)); A = np.zeros((n, n)); i = np.arange(n)
    history, stable, prev = [], 0, None
    for it in range(max_iter):
        # responsibility
        AS = A + S
        k = AS.argmax(1); first = AS[i, k]
        AS[i, k] = -np.inf; second = AS.max(1)
        Rn = S - first[:, None]; Rn[i, k] = S[i, k] - second
        R = damping * R + (1 - damping) * Rn
        # availability
        Rp = np.maximum(R, 0); np.fill_diagonal(Rp, R.diagonal())
        An = Rp.sum(0) - Rp; d = An.diagonal().copy()
        An = np.minimum(An, 0); np.fill_diagonal(An, d)
        A = damping * A + (1 - damping) * An
        E = A + R
        history.append(E.copy())
        ex = E.diagonal() > 0
        stable = stable + 1 if prev is not None and (ex == prev).all() else 0
        prev = ex
        if stable >= convergence_iter:
            break
    return history

history = ap_history(X)

def make_traces(E):
    score = E.diagonal()                    # >0 — точка центр
    target = E.argmax(1)                    # кого точка выбирает лидером
    lx, ly, lz = [], [], []
    for j, t in enumerate(target):
        if t != j:
            lx += [X[j, 0], X[t, 0], None]
            ly += [X[j, 1], X[t, 1], None]
            lz += [X[j, 2], X[t, 2], None]
    lines = go.Scatter3d(x=lx, y=ly, z=lz, mode='lines',
                         line=dict(color='rgba(0,0,0,0.35)', width=2), hoverinfo='skip')
    points = go.Scatter3d(x=X[:, 0], y=X[:, 1], z=X[:, 2], mode='markers',
                          marker=dict(size=[8 if s > 0 else 3 for s in score],
                                      color=score, cmin=-2, cmax=2,
                                      colorscale=[[0, 'green'], [0.5, 'black'], [1, 'red']]))
    return [lines, points]

fig = go.Figure(data=make_traces(history[0]),
                frames=[go.Frame(data=make_traces(E), name=str(t)) for t, E in enumerate(history)])

fig.update_layout(
    height=700, margin=dict(l=0, r=0, t=0, b=0), showlegend=False,
    sliders=[dict(steps=[dict(label=str(t + 1), method='animate',
                              args=[[str(t)], dict(mode='immediate', frame=dict(duration=0))])
                         for t in range(len(history))],
                  currentvalue=dict(prefix='Итерация: '))]
)
fig.show()